# Paper 1 — Validation Controls

Before drafting the paper from the main detection notebook (`qwen36_forward_planning_detection.ipynb`), we run these controls to rule out trivial explanations:

## What we're checking

| Control | Question | Expected if planning is REAL |
|---|---|---|
| **BOW baseline (early)** | Can bag-of-words of early response tokens predict letter? | **BOW <= 30%** (well below 52% from activations) |
| **BOW baseline (late)** | Can BOW of last 50 tokens predict letter? | **BOW ≈ chance** (just formatting tokens) |
| **Random-window BOW** | Can BOW of 100 RANDOM response tokens predict letter? | Similar to early BOW — if higher, leak is positional |
| **Token-level inspection** | What do early tokens literally say for letter-A rollouts vs letter-B? | **No obvious text signal** |
| **Seed robustness** | Does the 52% acc hold across 5 random splits? | **All seeds > 45%** |
| **Mechanistic vs semantic** | LogReg on activations vs LogReg on BOW — who wins? | **Activations win by > 20pp** |

If ALL pass → planning claim is scientifically solid, go to write.

If BOW comes close to 50% → text leak suspected, may need per-token analysis or different framing.

## Cell 1 — Setup (reuse artifacts if possible)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub==1.5.0', 'safetensors',
                'scikit-learn', 'scipy', 'matplotlib', 'seaborn', 'tqdm',
                'transformers', 'hf_transfer'], check=True)

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except: pass

import os, json, time, pickle
from pathlib import Path
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
OUT = Path('/content/planning')
OUT.mkdir(exist_ok=True)

# Reuse activations if available (from main notebook)
ACT_PATH = OUT / 'activations.npz'
if ACT_PATH.exists():
    print(f'\u2705 Loading cached activations from {ACT_PATH}')
    data = np.load(ACT_PATH, allow_pickle=True)
    labels_letter = data['labels_letter']
    labels_correct = data['labels_correct']
    early_pools = {w: {li: data[f'early_w{w}_L{li}'] for li in [11, 17, 23]} for w in [10, 25, 50, 100, 150]}
    late_pools = {li: data[f'late_L{li}'] for li in [11, 17, 23]}
    print(f'  {len(labels_letter)} rollouts loaded')
else:
    print('\u26a0\ufe0f  cache not found. Run main detection notebook first.')
    raise SystemExit

## Cell 2 — Extract response TEXT (not just activations) from Stage B

We need the actual text tokens of responses to run BOW baselines.
Stage B shards contain `response_tokens` in metadata.

In [ ]:
from huggingface_hub import snapshot_download
from safetensors import safe_open
from transformers import AutoTokenizer
from tqdm.auto import tqdm

corpus_path = snapshot_download(
    'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    repo_type='dataset', cache_dir='/content/cache')
shard_dir = Path(corpus_path) / 'shards'
shards = sorted(shard_dir.glob('q*.safetensors'))
print(f'{len(shards)} shards')

# Tokenizer for decoding
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3.6-35B-A3B', trust_remote_code=True)

# Extract response tokens (integer IDs) for each rollout — same filter as main notebook
MIN_LEN = 200
response_tokens_all = []   # list of lists of token ids (response only)
response_texts_all = []    # decoded texts
for shard in tqdm(shards, desc='text'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rollouts = json.loads(meta['rollouts'])
        rts = json.loads(meta['response_tokens'])
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN:
                continue
            response_tokens_all.append(rts[r_idx])
            response_texts_all.append(tok.decode(rts[r_idx], skip_special_tokens=True))

print(f'\u2705 {len(response_tokens_all)} responses extracted (should match {len(labels_letter)})')
assert len(response_tokens_all) == len(labels_letter), 'count mismatch — reproduce filter'

## Cell 3 — CONTROL 1: BOW baselines

Build three BOW classifiers from early/late/random-window TOKEN IDs (integer bag-of-token-ids):
- If early BOW approaches activation accuracy, the signal is textual, not mechanistic.
- If early BOW << activation, our claim survives.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

letter_to_int = {l: i for i, l in enumerate(sorted(set(labels_letter)))}
y = np.array([letter_to_int[l] for l in labels_letter])
n_classes = len(letter_to_int)
chance = 1.0 / n_classes

idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)

def tokens_to_docs(toks_list, window_type, window_size, response_lens=None):
    """Convert lists of token ids to space-separated token-id-strings for BOW."""
    docs = []
    for toks in toks_list:
        if window_type == 'early':
            w = toks[:window_size]
        elif window_type == 'late':
            w = toks[-window_size:]
        elif window_type == 'random':
            if len(toks) <= window_size:
                w = toks
            else:
                start = np.random.randint(0, len(toks) - window_size)
                w = toks[start:start+window_size]
        else:
            raise ValueError(window_type)
        docs.append(' '.join(str(t) for t in w))
    return docs

def bow_classify(docs_train, y_train, docs_test, y_test):
    vec = CountVectorizer(token_pattern=r'\S+')
    X_tr = vec.fit_transform(docs_train)
    X_te = vec.transform(docs_test)
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(X_tr, y_train)
    preds = lr.predict(X_te)
    return accuracy_score(y_test, preds), balanced_accuracy_score(y_test, preds)

results_bow = []
for window_type, window_size in [
    ('early', 10), ('early', 50), ('early', 100), ('early', 150),
    ('late', 50), ('random', 100),
]:
    np.random.seed(42)  # for reproducible random window
    docs = tokens_to_docs(response_tokens_all, window_type, window_size)
    docs_train = [docs[i] for i in idx_train]
    docs_test = [docs[i] for i in idx_test]
    acc, bal = bow_classify(docs_train, y[idx_train], docs_test, y[idx_test])
    results_bow.append(dict(type=window_type, size=window_size, acc=acc, bal_acc=bal))
    print(f'  BOW {window_type:8s} {window_size:3d} tokens: acc={acc:.3f}  bal_acc={bal:.3f}  ({acc/chance:.1f}× chance)')

print(f'\nActivation-based (best was early_w50_L17): 0.524')
print(f'Chance: {chance:.3f}')
print()
bow_early_best = max(r['acc'] for r in results_bow if r['type']=='early')
gap_vs_activation = 0.524 - bow_early_best
print(f'Best BOW early: {bow_early_best:.3f}')
print(f'Gap activation − BOW: {gap_vs_activation:+.3f} pp')

if gap_vs_activation > 0.15:
    print('  \u2705 Activations significantly beat BOW → mechanistic signal is real')
elif gap_vs_activation > 0.05:
    print('  \u26a0\ufe0f Modest gap — mechanistic adds value but text has signal too')
else:
    print('  \u274c BOW is competitive with activations → likely text leak, claim weakened')

## Cell 4 — CONTROL 2: Token-level inspection

Peek at what's literally in the first 100 tokens for correct/letter-A vs correct/letter-B responses. If the model literally writes "A" early, we need to handle that carefully.

In [ ]:
print('=== SAMPLE EARLY TEXT BY PREDICTED LETTER (first 300 chars) ===\n')
for target_letter in ['A', 'B', 'C']:
    matches = [i for i in range(len(labels_letter)) if labels_letter[i] == target_letter and labels_correct[i]]
    print(f'--- Predicted {target_letter} (correct, n={len(matches)}) — 3 samples ---')
    for i in matches[:3]:
        txt = response_texts_all[i]
        first_chars = txt[:300].replace('\n', ' ⏎ ')
        print(f'  [{i}] {first_chars}...')
    print()

# Check: does any letter appear in first 100 chars that matches the final answer?
print('=== LETTER LEAKAGE CHECK (in first 100 chars) ===\n')
import re
leak_counts = {l: {'total': 0, 'has_own_letter_early': 0, 'has_other_letter_early': 0} for l in letter_to_int}
for i, pred_letter in enumerate(labels_letter):
    early_text = response_texts_all[i][:100]
    # Find ANY letter pattern (e.g., "A.", "option A", "(A)", "A)")
    letters_mentioned = set(re.findall(r'\b([A-J])\b', early_text))
    leak_counts[pred_letter]['total'] += 1
    if pred_letter in letters_mentioned:
        leak_counts[pred_letter]['has_own_letter_early'] += 1
    if letters_mentioned - {pred_letter}:
        leak_counts[pred_letter]['has_other_letter_early'] += 1

print(f'{"Letter":<8} {"Total":>8} {"Has_own_early":>15} {"Has_other_early":>17}')
for l, c in sorted(leak_counts.items()):
    if c['total'] == 0: continue
    own_pct = 100 * c['has_own_letter_early'] / c['total']
    other_pct = 100 * c['has_other_letter_early'] / c['total']
    print(f'{l:<8} {c["total"]:>8} {c["has_own_letter_early"]:>10} ({own_pct:4.1f}%) {c["has_other_letter_early"]:>10} ({other_pct:4.1f}%)')

print('\nInterpretation:')
print('  If "has_own_letter_early" is LOW (<30%) and OTHER letter mentions are HIGH,')
print('  then the model is discussing options generally, not preannouncing its answer.')

## Cell 5 — CONTROL 3: Seed robustness

Re-run best config (early_w50_L17) under 5 different train/test splits. Accuracy should be stable (±5pp).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = early_pools[50][17]
seeds = [0, 7, 42, 123, 999]
seed_accs = []
seed_gaps = []  # correct vs wrong

for seed in seeds:
    idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=seed)
    sc = StandardScaler().fit(X[idx_tr])
    Xt = sc.transform(X[idx_tr]); Xv = sc.transform(X[idx_te])
    pca = PCA(n_components=128, random_state=seed).fit(Xt)
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=seed).fit(pca.transform(Xt), y[idx_tr])
    y_pred = lr.predict(pca.transform(Xv))
    acc = accuracy_score(y[idx_te], y_pred)
    # correct vs wrong gap
    corr_mask = labels_correct[idx_te] == 1
    acc_c = accuracy_score(y[idx_te][corr_mask], y_pred[corr_mask]) if corr_mask.sum() else 0
    acc_w = accuracy_score(y[idx_te][~corr_mask], y_pred[~corr_mask]) if (~corr_mask).sum() else 0
    seed_accs.append(acc)
    seed_gaps.append(acc_c - acc_w)
    print(f'  seed={seed}: acc={acc:.3f}  gap(correct-wrong)={acc_c-acc_w:+.3f}  (c={acc_c:.3f}, w={acc_w:.3f})')

print(f'\nacc mean±std: {np.mean(seed_accs):.3f} ± {np.std(seed_accs):.3f}')
print(f'gap mean±std: {np.mean(seed_gaps):+.3f} ± {np.std(seed_gaps):.3f}')

if np.std(seed_accs) < 0.03:
    print('  \u2705 Highly stable — results robust to split choice')
elif np.std(seed_accs) < 0.06:
    print('  \u2705 Reasonably stable')
else:
    print('  \u26a0\ufe0f High variance across seeds — result may be fragile')

## Cell 6 — CONTROL 4: Stratified shuffle baseline

Shuffle labels, keep activations — classifier should collapse to chance. If not, there's data leakage via preprocessing (scaler/PCA).

In [ ]:
X = early_pools[50][17]
np.random.seed(42)
y_shuf = np.random.permutation(y)

idx_tr, idx_te = train_test_split(np.arange(len(y_shuf)), test_size=0.2, stratify=y_shuf, random_state=42)
sc = StandardScaler().fit(X[idx_tr])
pca = PCA(n_components=128, random_state=42).fit(sc.transform(X[idx_tr]))
lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(pca.transform(sc.transform(X[idx_tr])), y_shuf[idx_tr])
acc_shuf = accuracy_score(y_shuf[idx_te], lr.predict(pca.transform(sc.transform(X[idx_te]))))

print(f'Shuffled labels baseline: {acc_shuf:.3f}  (should be ≈ chance {chance:.3f})')
if abs(acc_shuf - chance) < 0.03:
    print('  \u2705 Clean — no preprocessing leak')
elif acc_shuf < chance + 0.07:
    print('  \u2705 Mostly clean, small noise')
else:
    print('  \u274c Preprocessing leak — PCA or scaler leaking label info')

## Cell 7 — Final validation summary

In [ ]:
print('='*70)
print('PAPER 1 — VALIDATION SUMMARY')
print('='*70)

main_acc = 0.524  # from main notebook
bow_best = max(r['acc'] for r in results_bow if r['type']=='early')
bow_gap = main_acc - bow_best

checks = [
    ('Activation vs BOW gap', f'{bow_gap:+.3f}', bow_gap > 0.15, '> +0.150 (mechanistic clearly beats textual)'),
    ('Seed stability (std)', f'{np.std(seed_accs):.3f}', np.std(seed_accs) < 0.05, '< 0.05 (robust)'),
    ('Correct-vs-wrong gap (seed mean)', f'{np.mean(seed_gaps):+.3f}', np.mean(seed_gaps) > 0.30, '> +0.30 (planning load-bearing)'),
    ('Shuffled labels', f'{acc_shuf:.3f}', abs(acc_shuf - chance) < 0.05, f'≈ {chance:.3f} (no preprocessing leak)'),
]

print(f'\n{"Check":<35} {"Value":>10}   {"Pass?":<6}  Criterion')
print('-'*70)
for name, val, passed, crit in checks:
    tag = '\u2705' if passed else '\u274c'
    print(f'{name:<35} {val:>10}   {tag}      {crit}')

all_pass = all(c[2] for c in checks)
print()
if all_pass:
    print('  \u2b50 ALL CONTROLS PASSED — paper claim is scientifically solid')
    print('  Next: intervention experiments (B200) to add causality section')
else:
    failed = [c[0] for c in checks if not c[2]]
    print(f'  \u26a0\ufe0f FAILED: {failed}')
    print('  Need to address these before writing paper')

report = dict(
    main_activation_acc=main_acc,
    bow_best_early=bow_best,
    bow_vs_activation_gap=float(bow_gap),
    seed_accs=[float(a) for a in seed_accs],
    seed_gaps=[float(g) for g in seed_gaps],
    seed_acc_mean=float(np.mean(seed_accs)),
    seed_acc_std=float(np.std(seed_accs)),
    shuffled_labels_acc=float(acc_shuf),
    bow_results=results_bow,
    leakage_stats={l: c for l, c in leak_counts.items()},
    all_controls_passed=all_pass,
)
with open(OUT / 'validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'\n\u2705 saved {OUT}/validation_report.json')